# Explanation Notebook for ScenarioModel2
*Model Type:* **MLPClassifier**
*User:* UserDebug (Expertise: Advanced, Formats: [plainText, visual], Details: medium, Purpose: debugging)
*Explanation Method:* LIME


## Training the MLPClassifier Model
We train a **MLPClassifier** model on the provided dataset.


In [ ]:
import warnings; warnings.filterwarnings('ignore')
!pip install -q scikit-learn pandas lime matplotlib
import pandas as pd
from sklearn.model_selection import train_test_split
df = pd.read_csv(r"data/breast_cancer.csv")
y = df['target']
X = df.drop(columns=['target']).values
try:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
    print('[Info] Used stratified split to preserve class proportions in train/test.')
except Exception:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
    print('[Info] Used non-stratified split (stratification not applicable).')
class Dummy: pass
data = Dummy(); data.feature_names = list(df.drop(columns=['target']).columns)
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
base_model = MLPClassifier(max_iter=1000, random_state=0)
model = make_pipeline(StandardScaler(), base_model)
print('[Info] Normalization enabled by training policy: using StandardScaler in a pipeline.')
model.fit(X_train, y_train)
print('Accuracy: MLPClassifier = ' + format(model.score(X_test, y_test), '.2f'))


## Explaining the model using LIME
LIME explains individual predictions by fitting a simple local surrogate model around the instance.


In [ ]:
import re, numpy as np, pandas as pd, matplotlib.pyplot as plt

def _get_feature_names():
    try:
        names = list(getattr(data, 'feature_names', []))
        if names and len(names) == X_train.shape[1]:
            return names
    except Exception:
        pass
    if 'df' in globals():
        cols = [c for c in df.columns if c.lower() not in ('label','target','y')]
        if len(cols) == X_train.shape[1]:
            return cols
    return [f'f{j}' for j in range(X_train.shape[1])]

RAW_NAMES = _get_feature_names()
def _humanize(name):
    s = name.replace('_',' ').strip()
    s = re.sub(r'(?<!^)(?=[A-Z])',' ', s)
    s = re.sub(r'\s+',' ', s).strip()
    return ' '.join([w.upper() if w.lower() in {'svm','pdp','ice','api','url'} else w.capitalize() for w in s.split(' ')])
HUMAN = [_humanize(n) for n in RAW_NAMES]

max_display = 6
table_rows = 10
curve_resolution = 24
ice_instances = 10
metric_digits = 4
figure_height = 4.6
line_width = 2.0
pdp_sample_rows = 8
ice_detailed_instances = 3
sampled_curve_points = 7
def _choose_instance_debugging():
    try:
        y_pred = model.predict(X_test)
        wrong = np.where(y_pred != y_test)[0]
        if len(wrong) > 0:
            return int(wrong[0]), 'misclassified'
    except Exception:
        pass
    try:
        if hasattr(model, 'predict_proba'):
            proba = model.predict_proba(X_test)
            conf = np.max(proba, axis=1)
            return int(np.argmin(conf)), 'lowest-confidence'
    except Exception:
        pass
    return 0, 'default'

i, reason = _choose_instance_debugging()
print('[Debugging] Selected instance index = ' + str(i) + ' (reason: ' + str(reason) + ')')
print('\n[Debugging diagnostics]')
print('These diagnostics help inspect where the model succeeds, where it fails, and how errors are distributed.')
from sklearn.metrics import confusion_matrix, classification_report
y_pred = model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
print('Confusion matrix (rows = true labels, columns = predicted labels):')
print(cm)
print('Interpretation: each cell counts how many test examples of one true class were assigned to a predicted class.')
print('The diagonal corresponds to correct predictions; off-diagonal cells reveal the most common confusions.')
cm_sum = cm.sum(axis=1, keepdims=True)
cm_norm = np.divide(cm, cm_sum, where=(cm_sum != 0))
print('Normalized confusion matrix:')
print(np.round(cm_norm, metric_digits))
print('Interpretation: each row is normalized to sum to 1.0, so values can be read as per-class proportions rather than raw counts.')
plt.figure(figsize=(6, figure_height))
plt.imshow(cm, interpolation='nearest')
plt.title('Confusion matrix')
plt.colorbar()
tick_marks = np.arange(cm.shape[0])
plt.xticks(tick_marks, tick_marks)
plt.yticks(tick_marks, tick_marks)
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.tight_layout(); plt.show()
plt.figure(figsize=(6, figure_height))
plt.imshow(cm_norm, interpolation='nearest')
plt.title('Normalized confusion matrix')
plt.colorbar()
tick_marks = np.arange(cm_norm.shape[0])
plt.xticks(tick_marks, tick_marks)
plt.yticks(tick_marks, tick_marks)
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.tight_layout(); plt.show()
print('Classification report:')
print('Precision measures how often a predicted class is correct; recall measures how much of the true class is recovered; F1 balances both.')
print(classification_report(y_test, y_pred, digits=metric_digits))
print('\n[LIME overview]')
print('LIME explains one prediction by fitting a simple surrogate model around the selected instance.')
import lime.lime_tabular
explainer = lime.lime_tabular.LimeTabularExplainer(X_train, feature_names=HUMAN, class_names=['negative','positive'], mode='classification')
exp = explainer.explain_instance(X_test[i], model.predict_proba, num_features=max_display)
pairs = exp.as_list()
def _parse(rule):
    m = re.match(r'(.+?)\s*(<=|>=|<|>)\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)', rule)
    if not m:
        return {'raw_rule': rule, 'feature_label': rule, 'cmp': '', 'thr': None}
    left, cmp_op, thr = m.group(1).strip(), m.group(2), float(m.group(3))
    try:
        idx = HUMAN.index(left)
        return {'raw_rule': rule, 'feature_label': HUMAN[idx], 'cmp': cmp_op, 'thr': thr}
    except ValueError:
        return {'raw_rule': rule, 'feature_label': left, 'cmp': cmp_op, 'thr': thr}
parsed = []
for text, w in pairs:
    info = _parse(text)
    info['weight'] = float(w)
    info['direction'] = 'supports class 1' if float(w) > 0 else 'supports class 0'
    info['abs_weight'] = abs(float(w))
    parsed.append(info)
parsed = sorted(parsed, key=lambda d: abs(d['weight']), reverse=True)[:max_display]
df_lime = pd.DataFrame(parsed)
print('\n[Plain-text explanation]')
print('This explanation focuses on a selected case to support debugging-oriented analysis.')
for _, r in df_lime.iterrows():
    print(' - rule=' + str(r['raw_rule']) + ' | weight=' + format(r['weight'], '.6f'))
print('\n[Visual explanation]')
print('How to read the plot: bar length shows the magnitude of local influence; use the table/plain-text output for the direction of influence.')
print('Debugging note: interpret these factors together with the diagnostics above to understand why the selected case was difficult or incorrect.')
plot_df = df_lime.sort_values('abs_weight')
plt.figure(figsize=(8, figure_height))
plt.barh(plot_df['feature_label'], plot_df['abs_weight'])
plt.title('LIME local explanation (debugging view)')
plt.xlabel('Absolute local influence')
plt.grid(axis='x', alpha=0.2)
plt.tight_layout(); plt.show()
